In [4]:
import os
import sys
import glob

path = os.path.abspath("")
splitted = path.split(os.sep)
user_independent = os.path.join(
    splitted[0] + os.sep, splitted[1], splitted[2], splitted[3]
)
src_path = os.path.join(user_independent, "GitHub", "Photoswitching")
sys.path.append(src_path)

import src.blinking as bl
import src.distributions as dist
import src.emissions as em
import src.fcs as fcs_p
import src.figure as fi
import src.fluorophores as fl
import src.formulas as fo
import src.miscellaneous as mi
import src.network as net
import src.simulation as si
import src.prediction as pr
import src.analysis as an
import src.transitions as tr
import src.routines as ro
import src.fitting as fit

import numpy as np
import pandas as pd

%load_ext autoreload
%autoreload 2

import warnings


def custom_warning_format(msg, category, filename, lineno, line=None):
    if line is None:
        import linecache

        line = linecache.getline(filename, lineno)
    return f"WARNING for line: {line} {msg} \n"


warnings.formatwarning = custom_warning_format

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
def prepare_transition_set_ofret(number_fluorophores, distance):
    fluorophores = fl.construct_fluorophores(
    name="cy5_dna", distance=distance, count=number_fluorophores, shape="square"
    )
    fluorophore_system = fl.FluorophoreSystem(fluorophores=fluorophores)

    transitions = fluorophore_system.load_transitions(
    summarize=True,
    irradiance=5,
    wavelength=640,
    bleaching=True,
    energy_transfer=True,
    dstorm=True,
    dstorm_parameters={'reducing_agent':'mea',
    'concentration':100,
    'ph':7.5},
    energy_transfer_parameters={'overwrite': {'off': [1, 0.0001]}, 
                                'exclude': ['s0']}
    )
    transition_set = tr.TransitionSet(transitions, fluorophore_system)
    transition_set.finalize()

    return transition_set


In [ ]:
distances = [3, 6, 9, 18]
for distance in distances:
    transition_set = prepare_transition_set_ofret(2, distance)
    fingerprint_data, bleaching_times_all_runs = \
        ro.fingerprint_analysis(transition_set, batch_size=80, batches=1, # 900 min
                                filepath=r"D:\simulation_data\oet_variations\r100%eff001%2f", 
                                filename=f"{distance}nm", seed=1, use_memmap=r"C:\Users\vie43sq\Desktop\memmaps\run_1")

### Reading the data

In [8]:
distances = [3, 6, 9, 18]
identifiers = [f"{distance}nm" for distance in distances]
bleaching_times_all = []
fingerprints_all = []

for _ in identifiers:
    fingerprints_all.append(pd.Series(np.zeros(300001), 
                                      np.round(np.linspace(0, 300, 300001), decimals=12), 
                                      dtype=np.int32))
folder_path = r"C:\Users\vie43sq\Desktop\Simulations\simulation_data\oet_variations\r100%eff001%2f"
for file in glob.glob(folder_path + "/*"):
    if file.endswith(".npy"):
        bleaching_times_all.append(np.load(file))
    elif file.endswith(".parquet"):
        for id in identifiers:
            if id in file:
                fingerprints_all[identifiers.index(id)] += pd.read_parquet(file).sum(axis=1)
                break

for i, fingerprint in enumerate(fingerprints_all):
    fingerprint = fingerprint.cumsum() / fingerprint.sum()
    fingerprints_all[i] = fingerprint

In [10]:
x = np.linspace(0, 300, 300001)
parameters_ps_fingerprint_cdf_fit_2f_all = []
for fingerprint in fingerprints_all:
    y = fingerprint.values
    result = fit.ps_fingerprint_cdf_fit_2f(x, y, maxiter=1000, popsize=50, disp=False)
    parameters_ps_fingerprint_cdf_fit_2f_all.append(result.x)

WARNING for line:                 self.H.update(self.x - self.x_prev, self.g - self.g_prev)
 delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations. 
WARNING for line:                 self.H.update(self.x - self.x_prev, self.g - self.g_prev)
 delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations. 


In [ ]:
folder_path = r"C:\Users\vie43sq\Desktop\Simulations\simulation_data\oet_variations\r100%eff001%2f\fitting_parameters"
for identifier, parameters in zip(identifiers, parameters_ps_fingerprint_cdf_fit_2f_all):
    np.save(os.path.join(folder_path, f'{identifier}.npy'), parameters)

: 